
# Strander Monthly Vibration RMS Anomaly Model

This notebook is designed for historian exports containing approximately **one sample per second** of:

- Eight overall vibration RMS measurements in mm/s
- Line speed
- Timestamp

It does **not** require a planetary/non-planetary tag.

## What the notebook does

1. Loads one month-long CSV.
2. Verifies and cleans timestamps and numeric data.
3. Identifies machine runs using line speed.
4. Excludes startup, shutdown, stopped, and bad-data periods.
5. Builds five-minute analysis windows.
6. Calculates slow condition-monitoring features from each sensor.
7. Removes the normal effect of line speed from vibration.
8. Detects two recurring operating regimes automatically.
9. Trains a separate anomaly detector for each regime.
10. Ranks suspicious windows and exports review files.
11. Saves models so later monthly files can be scored consistently.

The model is intended to identify unusual behavior and developing changes. A flagged window is not automatically a confirmed mechanical fault.


## 1. Install packages if needed

In [ ]:

# Uncomment and run only if these packages are unavailable:
# %pip install pandas numpy matplotlib scikit-learn joblib


## 2. Imports

In [ ]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
import joblib

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)



## 3. User configuration

Change only this section first.

The CSV should be in wide format, with one timestamp, one line-speed value, and eight vibration values per row.

Example:

```text
Timestamp,LineSpeed,MotorTop,MotorBottom,GearboxTop,GearboxBottom,MotorSideBearing,MotorSideBase,NonMotorSideBearing,NonMotorSideBase
2026-07-01 08:00:00,79.8,1.10,1.22,1.85,1.61,1.42,0.71,1.20,0.62
```


In [ ]:

CSV_PATH = Path("replace_with_monthly_export.csv")

TIMESTAMP_COLUMN = "Timestamp"
LINE_SPEED_COLUMN = "LineSpeed"

VIBRATION_COLUMNS = [
    "MotorTop",
    "MotorBottom",
    "GearboxTop",
    "GearboxBottom",
    "MotorSideBearing",
    "MotorSideBase",
    "NonMotorSideBearing",
    "NonMotorSideBase",
]

# Adjust this to a speed that reliably indicates genuine operation rather than creep.
RUNNING_SPEED_THRESHOLD = 5.0

# Exclude transient periods after startup and before shutdown.
STARTUP_EXCLUSION_SECONDS = 120
SHUTDOWN_EXCLUSION_SECONDS = 120

# Five-minute windows at approximately one sample per second.
WINDOW_SECONDS = 300

# Advance one minute between windows.
WINDOW_STEP_SECONDS = 60

# Require at least this fraction of expected samples in a window.
MIN_WINDOW_COMPLETENESS = 0.90

# Reject any window containing a timestamp gap larger than this.
MAX_INTERNAL_GAP_SECONDS = 5

# Number of physical operating regimes expected.
N_OPERATING_REGIMES = 2

# Smooth out very brief regime assignments.
MIN_REGIME_DURATION_WINDOWS = 5

# Isolation Forest contamination is the expected proportion of unusual
# training windows. Keep small because faults should be rare.
ISOLATION_CONTAMINATION = 0.005

# Persistence rule for exported alerts.
# Example: three flagged one-minute-step windows represents a sustained event.
MIN_CONSECUTIVE_ANOMALOUS_WINDOWS = 3

# Use this directory to save models and monthly results.
OUTPUT_DIR = Path("strander_monthly_model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Training mode:
# True  = fit new speed models, regime detector, and anomaly detectors.
# False = load previously saved models and score a new month.
TRAIN_NEW_MODEL = True


## 4. Load and validate the monthly CSV

In [ ]:

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found: {CSV_PATH.resolve()}\n"
        "Update CSV_PATH in the configuration cell."
    )

df = pd.read_csv(CSV_PATH)

required_columns = [TIMESTAMP_COLUMN, LINE_SPEED_COLUMN, *VIBRATION_COLUMNS]
missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing)
        + "\n\nAvailable CSV columns:\n"
        + "\n".join(f"  - {c}" for c in df.columns)
    )

df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN], errors="coerce")

for column in [LINE_SPEED_COLUMN, *VIBRATION_COLUMNS]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = (
    df.dropna(subset=[TIMESTAMP_COLUMN])
      .sort_values(TIMESTAMP_COLUMN)
      .drop_duplicates(subset=[TIMESTAMP_COLUMN], keep="last")
      .reset_index(drop=True)
)

print("Rows:", f"{len(df):,}")
print("Time range:", df[TIMESTAMP_COLUMN].min(), "to", df[TIMESTAMP_COLUMN].max())
display(df.head())


## 5. Data-quality report

In [ ]:

quality = pd.DataFrame({
    "missing_count": df[required_columns].isna().sum(),
    "missing_percent": 100 * df[required_columns].isna().mean(),
    "unique_values": df[required_columns].nunique(dropna=True),
})

display(quality)

dt = df[TIMESTAMP_COLUMN].diff().dt.total_seconds()

print("Median sample interval:", dt.median(), "seconds")
print("90th percentile interval:", dt.quantile(0.90), "seconds")
print("Largest timestamp gaps:")
display(
    pd.DataFrame({
        "timestamp": df[TIMESTAMP_COLUMN],
        "gap_seconds": dt,
    }).nlargest(15, "gap_seconds")
)

constant_or_nearly_constant = []
for sensor in VIBRATION_COLUMNS:
    if df[sensor].nunique(dropna=True) < 10:
        constant_or_nearly_constant.append(sensor)

if constant_or_nearly_constant:
    print("WARNING: these channels have fewer than 10 unique values:")
    print(constant_or_nearly_constant)



## 6. Resample to one-second intervals

The historian export may contain occasional duplicate or skipped seconds.

This section creates a one-second time grid and interpolates only short gaps. Longer gaps remain missing and are excluded from analysis windows.


In [ ]:

data = df.set_index(TIMESTAMP_COLUMN).sort_index()

full_index = pd.date_range(
    start=data.index.min().floor("s"),
    end=data.index.max().ceil("s"),
    freq="1s",
)

data = data.reindex(full_index)
data.index.name = TIMESTAMP_COLUMN

numeric_columns = [LINE_SPEED_COLUMN, *VIBRATION_COLUMNS]

# Fill only short gaps of up to 5 seconds.
data[numeric_columns] = data[numeric_columns].interpolate(
    method="time",
    limit=5,
    limit_direction="both",
)

data = data.reset_index()
data["is_running_raw"] = data[LINE_SPEED_COLUMN] > RUNNING_SPEED_THRESHOLD

print("Resampled rows:", f"{len(data):,}")
print("Running fraction:", round(data["is_running_raw"].mean(), 4))



## 7. Detect machine runs and exclude startup/shutdown

A machine run is a continuous period above the line-speed threshold.

The notebook excludes the configured number of seconds after startup and before shutdown from the normal baseline.


In [ ]:

state_change = data["is_running_raw"].ne(data["is_running_raw"].shift(fill_value=False))
data["state_group"] = state_change.cumsum()

data["run_id"] = np.nan
run_counter = 0

for _, group in data.groupby("state_group"):
    if bool(group["is_running_raw"].iloc[0]):
        run_counter += 1
        data.loc[group.index, "run_id"] = run_counter

data["seconds_from_run_start"] = np.nan
data["seconds_to_run_end"] = np.nan

for run_id, group in data.dropna(subset=["run_id"]).groupby("run_id"):
    idx = group.index
    start_time = group[TIMESTAMP_COLUMN].iloc[0]
    end_time = group[TIMESTAMP_COLUMN].iloc[-1]

    data.loc[idx, "seconds_from_run_start"] = (
        group[TIMESTAMP_COLUMN] - start_time
    ).dt.total_seconds().values

    data.loc[idx, "seconds_to_run_end"] = (
        end_time - group[TIMESTAMP_COLUMN]
    ).dt.total_seconds().values

data["steady_running"] = (
    data["is_running_raw"]
    & (data["seconds_from_run_start"] >= STARTUP_EXCLUSION_SECONDS)
    & (data["seconds_to_run_end"] >= SHUTDOWN_EXCLUSION_SECONDS)
)

run_summary = (
    data.dropna(subset=["run_id"])
        .groupby("run_id")
        .agg(
            start=(TIMESTAMP_COLUMN, "min"),
            end=(TIMESTAMP_COLUMN, "max"),
            duration_seconds=("run_id", "size"),
            mean_speed=(LINE_SPEED_COLUMN, "mean"),
            max_speed=(LINE_SPEED_COLUMN, "max"),
        )
)

display(run_summary.head(20))
print("Detected runs:", len(run_summary))
print("Steady-running rows:", f"{int(data['steady_running'].sum()):,}")


In [ ]:

plt.figure(figsize=(14, 4))
plt.plot(data[TIMESTAMP_COLUMN], data[LINE_SPEED_COLUMN], linewidth=0.6)
plt.axhline(RUNNING_SPEED_THRESHOLD, linestyle="--", label="Running threshold")
plt.title("Monthly Line Speed")
plt.xlabel("Time")
plt.ylabel("Line speed")
plt.legend()
plt.tight_layout()
plt.show()



## 8. Build five-minute condition-monitoring windows

Each window contains only steady-running data from one continuous run.

For each sensor, the notebook calculates:

- Mean
- Median
- Standard deviation
- Minimum and maximum
- Peak-to-peak range
- Linear trend slope
- Beginning-to-end change
- 90th percentile
- Short-term change variability

It also calculates sensor-to-sensor relationships and line-speed statistics.


In [ ]:

def linear_slope(values):
    values = np.asarray(values, dtype=float)
    x = np.arange(len(values), dtype=float)
    valid = np.isfinite(values)

    if valid.sum() < 3:
        return np.nan

    return np.polyfit(x[valid], values[valid], 1)[0]


def build_condition_windows(
    data,
    window_seconds=300,
    step_seconds=60,
    min_completeness=0.90,
    max_gap_seconds=5,
):
    rows = []

    steady = data[data["steady_running"]].copy()

    for run_id, run in steady.groupby("run_id"):
        run = run.sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)

        if len(run) < window_seconds:
            continue

        start_time = run[TIMESTAMP_COLUMN].iloc[0]
        last_time = run[TIMESTAMP_COLUMN].iloc[-1]
        current = start_time

        while current + pd.Timedelta(seconds=window_seconds - 1) <= last_time:
            end = current + pd.Timedelta(seconds=window_seconds - 1)
            block = run[
                (run[TIMESTAMP_COLUMN] >= current)
                & (run[TIMESTAMP_COLUMN] <= end)
            ].copy()

            expected = window_seconds
            completeness = len(block) / expected

            if completeness < min_completeness:
                current += pd.Timedelta(seconds=step_seconds)
                continue

            gaps = block[TIMESTAMP_COLUMN].diff().dt.total_seconds()
            if gaps.max() > max_gap_seconds:
                current += pd.Timedelta(seconds=step_seconds)
                continue

            if block[[LINE_SPEED_COLUMN, *VIBRATION_COLUMNS]].isna().any().any():
                current += pd.Timedelta(seconds=step_seconds)
                continue

            row = {
                "run_id": int(run_id),
                "window_start": current,
                "window_end": end,
                "window_center": current + pd.Timedelta(seconds=window_seconds / 2),
                "completeness": completeness,
                "speed_mean": block[LINE_SPEED_COLUMN].mean(),
                "speed_std": block[LINE_SPEED_COLUMN].std(),
                "speed_min": block[LINE_SPEED_COLUMN].min(),
                "speed_max": block[LINE_SPEED_COLUMN].max(),
                "speed_slope": linear_slope(block[LINE_SPEED_COLUMN]),
                "seconds_from_run_start": block["seconds_from_run_start"].median(),
            }

            sensor_means = {}

            for sensor in VIBRATION_COLUMNS:
                values = block[sensor].to_numpy(dtype=float)
                sensor_means[sensor] = np.mean(values)

                row[f"{sensor}_mean"] = np.mean(values)
                row[f"{sensor}_median"] = np.median(values)
                row[f"{sensor}_std"] = np.std(values)
                row[f"{sensor}_min"] = np.min(values)
                row[f"{sensor}_max"] = np.max(values)
                row[f"{sensor}_range"] = np.ptp(values)
                row[f"{sensor}_slope"] = linear_slope(values)
                row[f"{sensor}_delta"] = np.mean(values[-30:]) - np.mean(values[:30])
                row[f"{sensor}_p90"] = np.percentile(values, 90)
                row[f"{sensor}_diff_std"] = np.std(np.diff(values))

            # Spatial relationships across machine locations.
            eps = 1e-6
            row["ratio_motor_top_bottom"] = (
                sensor_means["MotorTop"] / (sensor_means["MotorBottom"] + eps)
            )
            row["ratio_gearbox_top_bottom"] = (
                sensor_means["GearboxTop"] / (sensor_means["GearboxBottom"] + eps)
            )
            row["ratio_motor_bearing_to_base"] = (
                sensor_means["MotorSideBearing"] / (sensor_means["MotorSideBase"] + eps)
            )
            row["ratio_nonmotor_bearing_to_base"] = (
                sensor_means["NonMotorSideBearing"] / (sensor_means["NonMotorSideBase"] + eps)
            )
            row["ratio_motor_to_nonmotor_bearing"] = (
                sensor_means["MotorSideBearing"] / (sensor_means["NonMotorSideBearing"] + eps)
            )

            row["motor_average"] = np.mean([
                sensor_means["MotorTop"],
                sensor_means["MotorBottom"],
            ])
            row["gearbox_average"] = np.mean([
                sensor_means["GearboxTop"],
                sensor_means["GearboxBottom"],
            ])
            row["bearing_average"] = np.mean([
                sensor_means["MotorSideBearing"],
                sensor_means["NonMotorSideBearing"],
            ])
            row["base_average"] = np.mean([
                sensor_means["MotorSideBase"],
                sensor_means["NonMotorSideBase"],
            ])
            row["bearing_minus_base"] = row["bearing_average"] - row["base_average"]
            row["gearbox_minus_motor"] = row["gearbox_average"] - row["motor_average"]

            total = sum(sensor_means.values()) + eps
            for sensor, value in sensor_means.items():
                row[f"{sensor}_share"] = value / total

            rows.append(row)
            current += pd.Timedelta(seconds=step_seconds)

    return pd.DataFrame(rows)


windows = build_condition_windows(
    data,
    window_seconds=WINDOW_SECONDS,
    step_seconds=WINDOW_STEP_SECONDS,
    min_completeness=MIN_WINDOW_COMPLETENESS,
    max_gap_seconds=MAX_INTERNAL_GAP_SECONDS,
)

if windows.empty:
    raise ValueError(
        "No valid steady-running windows were created. "
        "Check the speed threshold, startup/shutdown exclusions, and data completeness."
    )

print("Condition windows:", f"{len(windows):,}")
display(windows.head())



## 9. Fit expected vibration versus line speed

Each sensor's normal five-minute mean is predicted from:

- Mean line speed
- Speed variation
- Speed trend
- Time since startup

The difference between actual and expected vibration is called the **speed-adjusted residual**.

This prevents the automatic regime detector from merely separating high-speed and low-speed operation.


In [ ]:

SPEED_CONTEXT_COLUMNS = [
    "speed_mean",
    "speed_std",
    "speed_min",
    "speed_max",
    "speed_slope",
    "seconds_from_run_start",
]

speed_models = {}
speed_residual_columns = []

if TRAIN_NEW_MODEL:
    for sensor in VIBRATION_COLUMNS:
        target = f"{sensor}_mean"
        model = HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=20,
            l2_regularization=1.0,
            random_state=RANDOM_SEED,
        )
        model.fit(windows[SPEED_CONTEXT_COLUMNS], windows[target])
        predicted = model.predict(windows[SPEED_CONTEXT_COLUMNS])

        windows[f"{sensor}_expected"] = predicted
        windows[f"{sensor}_speed_residual"] = windows[target] - predicted

        speed_models[sensor] = model
        speed_residual_columns.append(f"{sensor}_speed_residual")

    joblib.dump(speed_models, OUTPUT_DIR / "speed_models.joblib")
else:
    speed_models = joblib.load(OUTPUT_DIR / "speed_models.joblib")

    for sensor in VIBRATION_COLUMNS:
        target = f"{sensor}_mean"
        predicted = speed_models[sensor].predict(windows[SPEED_CONTEXT_COLUMNS])

        windows[f"{sensor}_expected"] = predicted
        windows[f"{sensor}_speed_residual"] = windows[target] - predicted
        speed_residual_columns.append(f"{sensor}_speed_residual")

display(
    windows[
        ["window_center", "speed_mean"]
        + [f"{s}_mean" for s in VIBRATION_COLUMNS[:2]]
        + [f"{s}_expected" for s in VIBRATION_COLUMNS[:2]]
        + [f"{s}_speed_residual" for s in VIBRATION_COLUMNS[:2]]
    ].head()
)



## 10. Automatically detect the two physical operating regimes

The regime detector uses:

- Speed-adjusted vibration residuals
- Sensor energy shares
- Sensor-to-sensor ratios
- Differences between machine areas

It does not assume that one configuration always has higher vibration.


In [ ]:

REGIME_FEATURE_COLUMNS = (
    speed_residual_columns
    + [f"{sensor}_share" for sensor in VIBRATION_COLUMNS]
    + [
        "ratio_motor_top_bottom",
        "ratio_gearbox_top_bottom",
        "ratio_motor_bearing_to_base",
        "ratio_nonmotor_bearing_to_base",
        "ratio_motor_to_nonmotor_bearing",
        "bearing_minus_base",
        "gearbox_minus_motor",
    ]
)

if TRAIN_NEW_MODEL:
    regime_scaler = RobustScaler()
    regime_X = regime_scaler.fit_transform(windows[REGIME_FEATURE_COLUMNS])

    regime_model = GaussianMixture(
        n_components=N_OPERATING_REGIMES,
        covariance_type="full",
        n_init=20,
        random_state=RANDOM_SEED,
    )
    raw_labels = regime_model.fit_predict(regime_X)

    joblib.dump(regime_scaler, OUTPUT_DIR / "regime_scaler.joblib")
    joblib.dump(regime_model, OUTPUT_DIR / "regime_model.joblib")
else:
    regime_scaler = joblib.load(OUTPUT_DIR / "regime_scaler.joblib")
    regime_model = joblib.load(OUTPUT_DIR / "regime_model.joblib")

    regime_X = regime_scaler.transform(windows[REGIME_FEATURE_COLUMNS])
    raw_labels = regime_model.predict(regime_X)

windows["raw_regime"] = [f"regime_{label}" for label in raw_labels]
windows["regime_confidence"] = regime_model.predict_proba(regime_X).max(axis=1)

if len(np.unique(raw_labels)) > 1:
    print("Silhouette score:", round(silhouette_score(regime_X, raw_labels), 3))

print("Mean regime confidence:", round(windows["regime_confidence"].mean(), 3))
display(windows["raw_regime"].value_counts())


## 11. Smooth brief regime changes

In [ ]:

def smooth_regimes_by_run(frame, minimum_windows=5):
    frame = frame.copy()
    frame["detected_regime"] = frame["raw_regime"]

    for run_id, group in frame.groupby("run_id"):
        idx = list(group.sort_values("window_center").index)
        labels = frame.loc[idx, "raw_regime"].tolist()
        smoothed = labels.copy()

        start = 0
        while start < len(labels):
            end = start + 1
            while end < len(labels) and labels[end] == labels[start]:
                end += 1

            if end - start < minimum_windows:
                previous_label = smoothed[start - 1] if start > 0 else None
                next_label = labels[end] if end < len(labels) else None
                replacement = previous_label if previous_label is not None else next_label

                if replacement is not None:
                    for i in range(start, end):
                        smoothed[i] = replacement

            start = end

        frame.loc[idx, "detected_regime"] = smoothed

    return frame


windows = smooth_regimes_by_run(
    windows,
    minimum_windows=MIN_REGIME_DURATION_WINDOWS,
)

display(pd.crosstab(windows["raw_regime"], windows["detected_regime"]))


In [ ]:

plt.figure(figsize=(14, 5))
for regime, group in windows.groupby("detected_regime"):
    plt.scatter(
        group["window_center"],
        group["speed_mean"],
        s=10,
        alpha=0.6,
        label=regime,
    )

plt.title("Automatically Detected Operating Regimes")
plt.xlabel("Time")
plt.ylabel("Mean line speed")
plt.legend()
plt.tight_layout()
plt.show()



## 12. Compare the two detected baselines

Use this section to determine which cluster corresponds to planetary operation.

The anomaly detector works even if the clusters remain named `regime_0` and `regime_1`.


In [ ]:

comparison_columns = (
    ["speed_mean", "regime_confidence"]
    + [f"{sensor}_mean" for sensor in VIBRATION_COLUMNS]
    + speed_residual_columns
    + [
        "motor_average",
        "gearbox_average",
        "bearing_average",
        "base_average",
    ]
)

display(
    windows.groupby("detected_regime")[comparison_columns]
           .agg(["mean", "std", "median"])
)

for sensor in VIBRATION_COLUMNS:
    plt.figure(figsize=(7, 5))
    for regime, group in windows.groupby("detected_regime"):
        plt.scatter(
            group["speed_mean"],
            group[f"{sensor}_mean"],
            s=9,
            alpha=0.3,
            label=regime,
        )

    plt.title(f"{sensor}: Vibration RMS by Detected Regime")
    plt.xlabel("Mean line speed")
    plt.ylabel("Mean vibration RMS (mm/s)")
    plt.legend()
    plt.tight_layout()
    plt.show()



## 13. Train one anomaly detector per regime

Isolation Forest is used because:

- Fault labels are not required
- It works well with engineered condition-monitoring features
- It is less computationally expensive than a deep neural network
- Its results are easier to interpret for a first industrial model

The detector considers level, variation, trend, line-speed residuals, and relationships across all eight sensors.


In [ ]:

ANOMALY_FEATURE_COLUMNS = (
    SPEED_CONTEXT_COLUMNS
    + [f"{sensor}_{stat}" for sensor in VIBRATION_COLUMNS for stat in [
        "mean", "median", "std", "max", "range", "slope", "delta", "p90", "diff_std"
    ]]
    + speed_residual_columns
    + [f"{sensor}_share" for sensor in VIBRATION_COLUMNS]
    + [
        "ratio_motor_top_bottom",
        "ratio_gearbox_top_bottom",
        "ratio_motor_bearing_to_base",
        "ratio_nonmotor_bearing_to_base",
        "ratio_motor_to_nonmotor_bearing",
        "motor_average",
        "gearbox_average",
        "bearing_average",
        "base_average",
        "bearing_minus_base",
        "gearbox_minus_motor",
    ]
)

anomaly_assets = {}

if TRAIN_NEW_MODEL:
    for regime, group in windows.groupby("detected_regime"):
        scaler = RobustScaler()
        X = scaler.fit_transform(group[ANOMALY_FEATURE_COLUMNS])

        model = IsolationForest(
            n_estimators=500,
            contamination=ISOLATION_CONTAMINATION,
            max_samples="auto",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        model.fit(X)

        anomaly_assets[regime] = {
            "scaler": scaler,
            "model": model,
        }

    joblib.dump(anomaly_assets, OUTPUT_DIR / "anomaly_assets.joblib")
else:
    anomaly_assets = joblib.load(OUTPUT_DIR / "anomaly_assets.joblib")

windows["anomaly_score"] = np.nan
windows["is_anomaly_raw"] = False

for regime, group in windows.groupby("detected_regime"):
    if regime not in anomaly_assets:
        print(f"WARNING: no saved anomaly model exists for {regime}")
        continue

    scaler = anomaly_assets[regime]["scaler"]
    model = anomaly_assets[regime]["model"]

    X = scaler.transform(group[ANOMALY_FEATURE_COLUMNS])

    # Larger positive values mean more unusual.
    score = -model.score_samples(X)
    prediction = model.predict(X)

    windows.loc[group.index, "anomaly_score"] = score
    windows.loc[group.index, "is_anomaly_raw"] = prediction == -1

display(
    windows.groupby("detected_regime")["is_anomaly_raw"]
           .agg(["count", "sum", "mean"])
)



## 14. Apply persistence logic

A single unusual five-minute window can be caused by a product change, transient loading, or random variation.

The persistence rule promotes only consecutive anomalous windows into review events.


In [ ]:

windows = windows.sort_values("window_center").reset_index(drop=True)
windows["persistent_anomaly"] = False
windows["anomaly_event_id"] = np.nan

event_counter = 0

for run_id, group in windows.groupby("run_id"):
    idx = list(group.index)
    flags = windows.loc[idx, "is_anomaly_raw"].to_numpy(dtype=bool)

    start = 0
    while start < len(flags):
        if not flags[start]:
            start += 1
            continue

        end = start + 1
        while end < len(flags) and flags[end]:
            end += 1

        if end - start >= MIN_CONSECUTIVE_ANOMALOUS_WINDOWS:
            event_counter += 1
            event_indices = idx[start:end]
            windows.loc[event_indices, "persistent_anomaly"] = True
            windows.loc[event_indices, "anomaly_event_id"] = event_counter

        start = end

print("Persistent anomaly events:", event_counter)



## 15. Explain which sensors contributed most

This uses robust deviation within each regime to estimate which features were most unusual during each window.


In [ ]:

sensor_explanation_columns = [
    f"{sensor}_mean" for sensor in VIBRATION_COLUMNS
] + speed_residual_columns

for column in sensor_explanation_columns:
    windows[f"z_{column}"] = np.nan

for regime, group in windows.groupby("detected_regime"):
    for column in sensor_explanation_columns:
        median = group[column].median()
        mad = np.median(np.abs(group[column] - median))
        robust_scale = max(1.4826 * mad, 1e-6)

        windows.loc[group.index, f"z_{column}"] = (
            (group[column] - median).abs() / robust_scale
        )

z_columns = [f"z_{column}" for column in sensor_explanation_columns]

windows["dominant_unusual_feature"] = (
    windows[z_columns]
    .idxmax(axis=1)
    .str.replace("z_", "", regex=False)
)

windows["dominant_feature_score"] = windows[z_columns].max(axis=1)


## 16. Review the most suspicious windows

In [ ]:

review_columns = [
    "window_center",
    "run_id",
    "detected_regime",
    "regime_confidence",
    "speed_mean",
    "anomaly_score",
    "is_anomaly_raw",
    "persistent_anomaly",
    "anomaly_event_id",
    "dominant_unusual_feature",
    "dominant_feature_score",
] + [f"{sensor}_mean" for sensor in VIBRATION_COLUMNS]

ranked = windows.sort_values("anomaly_score", ascending=False)

display(ranked[review_columns].head(100))


In [ ]:

plt.figure(figsize=(14, 5))
plt.plot(
    windows["window_center"],
    windows["anomaly_score"],
    linewidth=0.7,
    label="Anomaly score",
)

persistent = windows[windows["persistent_anomaly"]]
plt.scatter(
    persistent["window_center"],
    persistent["anomaly_score"],
    s=20,
    label="Persistent anomaly",
)

plt.title("Monthly Strander Anomaly Score")
plt.xlabel("Time")
plt.ylabel("Isolation score — higher is more unusual")
plt.legend()
plt.tight_layout()
plt.show()


## 17. Create an event-level summary

In [ ]:

event_rows = []

for event_id, group in windows.dropna(subset=["anomaly_event_id"]).groupby("anomaly_event_id"):
    top = group.sort_values("anomaly_score", ascending=False).iloc[0]

    event_rows.append({
        "event_id": int(event_id),
        "start_time": group["window_start"].min(),
        "end_time": group["window_end"].max(),
        "duration_minutes": (
            group["window_end"].max() - group["window_start"].min()
        ).total_seconds() / 60,
        "run_id": int(group["run_id"].mode().iloc[0]),
        "detected_regime": group["detected_regime"].mode().iloc[0],
        "mean_line_speed": group["speed_mean"].mean(),
        "maximum_anomaly_score": group["anomaly_score"].max(),
        "most_unusual_feature": top["dominant_unusual_feature"],
        "feature_deviation_score": top["dominant_feature_score"],
        "window_count": len(group),
    })

events = pd.DataFrame(event_rows)

if len(events):
    events = events.sort_values("maximum_anomaly_score", ascending=False)
    display(events)
else:
    print("No persistent anomaly events met the configured rule.")



## 18. Export results

The output files include:

- All five-minute windows and scores
- The highest-ranked windows
- Persistent anomaly events
- Machine-run summary
- Saved models when training mode is enabled


In [ ]:

month_name = pd.to_datetime(data[TIMESTAMP_COLUMN].min()).strftime("%Y-%m")

windows.to_csv(
    OUTPUT_DIR / f"{month_name}_all_window_scores.csv",
    index=False,
)

ranked.head(500).to_csv(
    OUTPUT_DIR / f"{month_name}_top_500_windows.csv",
    index=False,
)

run_summary.to_csv(
    OUTPUT_DIR / f"{month_name}_run_summary.csv",
)

if len(events):
    events.to_csv(
        OUTPUT_DIR / f"{month_name}_persistent_anomaly_events.csv",
        index=False,
    )

configuration = {
    "timestamp_column": TIMESTAMP_COLUMN,
    "line_speed_column": LINE_SPEED_COLUMN,
    "vibration_columns": VIBRATION_COLUMNS,
    "running_speed_threshold": RUNNING_SPEED_THRESHOLD,
    "startup_exclusion_seconds": STARTUP_EXCLUSION_SECONDS,
    "shutdown_exclusion_seconds": SHUTDOWN_EXCLUSION_SECONDS,
    "window_seconds": WINDOW_SECONDS,
    "window_step_seconds": WINDOW_STEP_SECONDS,
    "regime_feature_columns": REGIME_FEATURE_COLUMNS,
    "anomaly_feature_columns": ANOMALY_FEATURE_COLUMNS,
}

with open(OUTPUT_DIR / "model_configuration.json", "w") as f:
    json.dump(configuration, f, indent=2)

print("Results written to:", OUTPUT_DIR.resolve())



## 19. Optional maintenance-event comparison

Create a CSV with at least:

```text
event_time,event_type,description
2026-07-18 14:32:00,Drive Overcurrent,Drive fault during production
```

This section attaches the nearest model window and checks the preceding anomaly history.


In [ ]:

MAINTENANCE_EVENTS_CSV = Path("optional_maintenance_events.csv")
EVENT_TIME_COLUMN = "event_time"

if MAINTENANCE_EVENTS_CSV.exists():
    maintenance = pd.read_csv(MAINTENANCE_EVENTS_CSV)
    maintenance[EVENT_TIME_COLUMN] = pd.to_datetime(
        maintenance[EVENT_TIME_COLUMN],
        errors="coerce",
    )
    maintenance = maintenance.dropna(subset=[EVENT_TIME_COLUMN]).sort_values(EVENT_TIME_COLUMN)

    aligned = pd.merge_asof(
        maintenance,
        windows.sort_values("window_center"),
        left_on=EVENT_TIME_COLUMN,
        right_on="window_center",
        direction="nearest",
        tolerance=pd.Timedelta("30 minutes"),
    )

    lookback_rows = []

    for _, event in maintenance.iterrows():
        row = event.to_dict()
        event_time = event[EVENT_TIME_COLUMN]

        for hours in [1, 6, 24, 72, 168]:
            prior = windows[
                (windows["window_center"] >= event_time - pd.Timedelta(hours=hours))
                & (windows["window_center"] < event_time)
            ]

            row[f"max_anomaly_score_prior_{hours}h"] = (
                prior["anomaly_score"].max() if len(prior) else np.nan
            )
            row[f"persistent_windows_prior_{hours}h"] = (
                int(prior["persistent_anomaly"].sum()) if len(prior) else 0
            )

        lookback_rows.append(row)

    lookback = pd.DataFrame(lookback_rows)

    display(aligned.head())
    display(lookback.head())

    aligned.to_csv(
        OUTPUT_DIR / f"{month_name}_maintenance_nearest_scores.csv",
        index=False,
    )
    lookback.to_csv(
        OUTPUT_DIR / f"{month_name}_maintenance_lookback.csv",
        index=False,
    )
else:
    print("Optional maintenance-events file not found. Skipping this section.")



## 20. Suggested monthly workflow

### Initial baseline

For the first model, use a month believed to contain mostly normal operation:

```python
TRAIN_NEW_MODEL = True
```

Review the detected regimes and highest-ranked anomalies. If the month contains a known major failure period, remove that period or choose a cleaner baseline month.

### Later months

Place the next monthly CSV in the notebook folder and change:

```python
CSV_PATH = Path("next_month.csv")
TRAIN_NEW_MODEL = False
```

This loads the previously trained speed, regime, and anomaly models so scores remain comparable from month to month.

### Periodic retraining

Do not automatically retrain every month. Retrain only when:

- Sensor hardware or mounting changes
- The machine undergoes a major mechanical rebuild
- Product mix changes substantially
- Review confirms the existing baseline no longer represents healthy operation
- Several clean months can be combined into a stronger training baseline



## 21. Important limitations

Because the historian stores one overall RMS value per second, this model cannot identify bearing defect frequencies or gear-mesh frequencies. It detects changes in vibration severity, trends, persistence, and relationships across the eight sensor locations.

Useful future inputs would include:

- Drive torque
- Motor current
- Motor or gearbox temperature
- Product recipe
- Product diameter
- Payoff loading
- Drive fault timestamps
- Maintenance work orders
- Known dates when the physical gearing was changed

Those signals can be added later without replacing the basic monthly workflow.
